# Harmonic Content Analysis

The spectral centroid reported in `audio_processing.ipynb` is computed over the full band up to
Nyquist. For a note whose fundamental lies near 330 Hz, a centroid above 7 kHz places the centre of
gravity beyond the twentieth partial, so a natural question is whether the natural-to-bent centroid
difference reflects a redistribution of *harmonic* energy or simply a difference in broadband breath
noise.

This notebook separates the two. It splits each spectrum into a harmonic part and a noise floor,
recomputes the centroid on the harmonic part alone, and describes how energy is distributed across
frequency bands. It closes with two controls: clipping and the room noise floor.

## Setup

Frame size, hop size and sample rate are the same as in `audio_processing.ipynb`, so the spectra
here are directly comparable to the descriptors already published.

In [ ]:
import csv
import numpy as np
import wave

SR, FRAME_SIZE, HOP_SIZE = 48000, 1024, 256
WINDOW = np.hamming(FRAME_SIZE)
FREQ = np.fft.rfftfreq(FRAME_SIZE, 1 / SR)
BIN_WIDTH = FREQ[1]

AUDIO_DIR = '../data/audios_v2'
DESCRIPTORS = '../data/descriptors_v2.csv'

print(f'frequency resolution: {BIN_WIDTH:.1f} Hz per bin')

Essentia is not used here. The analysis needs the raw magnitude spectrum and a per-frame noise floor
estimate, neither of which is exposed by the descriptor algorithms, so the STFT is computed directly
and the recordings are read with the standard library. Recordings are 24 bit, which `wave` returns as
raw bytes, so the samples are assembled explicitly.

In [ ]:
def read_wav(path):
    """Read a PCM wav file into a float64 mono array in [-1, 1]."""
    with wave.open(path, 'rb') as w:
        sr, width, channels = w.getframerate(), w.getsampwidth(), w.getnchannels()
        raw = w.readframes(w.getnframes())
    if width == 3:
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        x = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16)
        x = np.where(x & 0x800000, x - 0x1000000, x).astype(np.float64) / 2 ** 23
    elif width == 2:
        x = np.frombuffer(raw, dtype='<i2').astype(np.float64) / 2 ** 15
    elif width == 4:
        x = np.frombuffer(raw, dtype='<i4').astype(np.float64) / 2 ** 31
    else:
        raise ValueError(f'unsupported sample width: {width}')
    if channels > 1:
        x = x.reshape(-1, channels).mean(axis=1)
    return sr, x


def stft_magnitude(x, start=2.0, end=5.0):
    """Magnitude spectra over the sustained window used for the centroid."""
    segment = x[int(start * SR):int(end * SR)]
    starts = range(0, len(segment) - FRAME_SIZE + 1, HOP_SIZE)
    return np.array([np.abs(np.fft.rfft(segment[i:i + FRAME_SIZE] * WINDOW)) for i in starts])


def recording_path(row):
    return (f"{AUDIO_DIR}/harp{row['harp']}_{row['material']}_"
            f"{row['method']}_{row['note']}_rep{row['rep']}.wav")


rows = list(csv.DictReader(open(DESCRIPTORS)))
for r in rows:
    r['f0'] = float(r['f0'])
    r['centroid'] = float(r['centroid'])
print(f'{len(rows)} recordings')

## Pipeline validation

Before drawing any conclusion from a derived quantity, the pipeline has to reproduce a quantity that
is already published. The centroid recomputed here is compared against the `centroid` column of
`descriptors_v2.csv`. Agreement also settles whether the published values are magnitude weighted or
power weighted.

In [ ]:
def centroid(mag, weight='magnitude', low=None, high=None):
    m = mag.copy()
    if low is not None:
        m[:, FREQ < low] = 0
    if high is not None:
        m[:, FREQ > high] = 0
    w = m if weight == 'magnitude' else m ** 2
    num, den = (w * FREQ).sum(1), w.sum(1)
    ok = den > 0
    return float(np.mean(num[ok] / den[ok])) if ok.any() else np.nan


spectra = {}
for r in rows:
    _, x = read_wav(recording_path(r))
    spectra[id(r)] = stft_magnitude(x)

published = np.array([r['centroid'] for r in rows])
for weight in ('magnitude', 'power'):
    v = np.array([centroid(spectra[id(r)], weight) for r in rows])
    err = np.median(np.abs(v - published) / published) * 100
    print(f'{weight:>9} weighting: correlation {np.corrcoef(published, v)[0, 1]:.4f}, '
          f'median error {err:.2f}%')

## Harmonic and noise components

A first attempt used a mask of a fixed number of bins around each multiple of $f_0$. That approach is
invalid at this resolution: with $f_0 \approx 330$ Hz and 46.9 Hz per bin, consecutive harmonics are
about seven bins apart, so a mask of five bins covers most of the spectrum and separates nothing.

The floor is therefore estimated with a running median along frequency, over a window wider than one
harmonic spacing. Peaks that rise above that floor are treated as harmonic energy and the floor
itself as the noise component.

In [ ]:
def median_floor(mag, width_bins):
    width_bins = int(width_bins) | 1  # force odd
    pad = width_bins // 2
    padded = np.pad(mag, ((0, 0), (pad, pad)), mode='edge')
    idx = np.arange(mag.shape[1])[:, None] + np.arange(width_bins)[None, :]
    return np.median(padded[:, idx], axis=2)


def harmonic_split(mag, f0):
    floor = median_floor(mag, round(2 * f0 / BIN_WIDTH))
    harmonic = np.clip(mag - floor, 0, None)
    return harmonic, floor


BANDS = [(0, 1000), (1000, 2000), (2000, 5000), (5000, 10000), (10000, 24000)]

for r in rows:
    mag = spectra[id(r)]
    harmonic, floor = harmonic_split(mag, r['f0'])
    total = (mag ** 2).sum()
    r['harmonic_fraction'] = float((harmonic ** 2).sum() / total)
    r['hnr_db'] = float(10 * np.log10((harmonic ** 2).sum() / max((floor ** 2).sum(), 1e-30)))
    num, den = (harmonic * FREQ).sum(1), harmonic.sum(1)
    ok = den > 0
    r['harmonic_centroid'] = float(np.mean(num[ok] / den[ok]))
    for low, high in BANDS:
        band = (FREQ >= low) & (FREQ < high)
        r[f'band_{low // 1000}_{high // 1000}'] = float((mag[:, band] ** 2).sum() / total)

## Results by emission method

Natural notes are split by airflow direction, because the two blow notes (E4, G4) and the two draw
notes (F4, A4) start from very different spectra.

In [ ]:
import statistics as st

groups = {
    'natural blow': [r for r in rows if r['method'] == 'natural' and r['note'] in ('E4', 'G4')],
    'natural draw': [r for r in rows if r['method'] == 'natural' and r['note'] in ('F4', 'A4')],
    'bend': [r for r in rows if r['method'] == 'bend'],
}
median = lambda g, k: st.median([r[k] for r in g])

print(f"{'':28}" + ''.join(f'{name:>16}' for name in groups))
for key, label in (('harmonic_fraction', 'harmonic fraction'),
                   ('hnr_db', 'harmonic-to-noise (dB)'),
                   ('harmonic_centroid', 'harmonic centroid (Hz)'),
                   ('centroid', 'full-band centroid (Hz)')):
    print(f'{label:28}' + ''.join(f'{median(g, key):16.2f}' for g in groups.values()))

print('\nenergy distribution by band (% of total)')
for low, high in BANDS:
    key = f'band_{low // 1000}_{high // 1000}'
    print(f'  {low // 1000:>2}-{high // 1000:>2} kHz{" " * 14}'
          + ''.join(f'{median(g, key) * 100:16.2f}' for g in groups.values()))

The signal is harmonic in every condition, and restricting the centroid to the harmonic component
*increases* the natural-to-bent ratio rather than reducing it. The difference between natural and
bent notes is therefore not an artefact of broadband breath noise.

What the band table shows is sharper than a shift of the centre of gravity: bending removes the high
frequency content almost entirely.

In [ ]:
print('suppression of energy above 5 kHz, by note')
print(f"{'note':6}{'natural':>10}{'bend':>10}{'factor':>10}")
for note in ('E4', 'F4', 'G4', 'A4'):
    nat = [r for r in rows if r['note'] == note and r['method'] == 'natural']
    ben = [r for r in rows if r['note'] == note and r['method'] == 'bend']
    a = median(nat, 'band_5_10') + median(nat, 'band_10_24')
    b = median(ben, 'band_5_10') + median(ben, 'band_10_24')
    print(f'{note:6}{a * 100:9.2f}%{b * 100:9.2f}%{a / b:9.0f}x')

above5 = lambda r: r['band_5_10'] + r['band_10_24']
bent = [r for r in rows if r['method'] == 'bend']
natural = [r for r in rows if r['method'] == 'natural']
print(f'\nrecordings with less than 0.5% of energy above 5 kHz:')
print(f'  bent    {sum(above5(r) < 0.005 for r in bent)} of {len(bent)}')
print(f'  natural {sum(above5(r) < 0.005 for r in natural)} of {len(natural)}')

## Controls

Two alternative explanations have to be excluded. High frequency content could be distortion
introduced when the interface gain was set, and the very low figures for bent notes could simply be
the noise floor of the room rather than a property of the instrument.

In [ ]:
peaks = {name: [] for name in groups}
for name, g in groups.items():
    for r in g:
        _, x = read_wav(recording_path(r))
        peaks[name].append(np.abs(x[2 * SR:5 * SR]).max())

print('peak amplitude in the analysis window (1.0 is full scale)')
for name, v in peaks.items():
    print(f'  {name:14} median {np.median(v):.3f}  max {max(v):.3f}  above 0.99: {sum(z > 0.99 for z in v)}')

For the noise floor, the session includes a 30 s recording of room silence made under the same gain
setting. Point `SILENCE` at it to compare band by band. The comparison shown in the manuscript places
the content of bent notes above 10 kHz well above that floor, which rules out the floor as the source
of the measured figures. Because bent notes are also the quieter of the two classes, any residual
noise contamination would inflate their high frequency fraction, so the suppression reported here is
conservative.

In [ ]:
SILENCE = None  # path to the 30 s room silence recording

if SILENCE:
    _, s = read_wav(SILENCE)
    silence = stft_magnitude(s)
    reference = [r for r in rows if r['method'] == 'bend'][0]
    bend_mag = spectra[id(reference)]
    for low, high in ((0, 1000), (5000, 10000), (10000, 24000)):
        band = (FREQ >= low) & (FREQ < high)
        a = (bend_mag[:, band] ** 2).sum()
        b = (silence[:, band] ** 2).sum()
        print(f'{low // 1000:>2}-{high // 1000:>2} kHz: {10 * np.log10(a / b):+6.1f} dB above room silence')